In [ ]:
import numpy as np
import cldnufuncs as cfunc
import matplotlib.pyplot as plt
# import spherical_bin as sp
# import window as win
# import pkfuncs as pfunc

In [ ]:
Nmax    =  256
Nr      =  8
LL      =  3.438
LL     *=  np.pi/(60.*180); 
ff      = .00025

In [ ]:
path = '/home/sukhdeep/Documents/Research/Bispectrum/Visibility_Bispectrum/Visibility_Correlation/asif_da/3_vcg/output_data/' # path to the Gridded visibilities.

# Ncores = 10
Nmg = 1 # Number of realizations for Mg
Nmodel = 1 # Number of realizations for Mg
# Nnoise = 1 # Number of realizations for noise.
# vrms = 1. # Jy

# inputs
nc = 1   #768 # nc is always used for total channels
NC = 1   # no. of channels to collapse
xn = 0   # no. of grids to exclude from from each sides of the gridded data.
# na = nc//NC # No. of channels available after collapse.

# header
headersize = (nc+3)*np.float64().itemsize
# hinfo = cfunc.read_header(str(path)+'GV-model-1000000240-1', headersize, nc)
hinfo = cfunc.read_header(str(path)+'UGV_data_1', headersize, nc)
hinfo = np.asarray(hinfo, dtype='object')
np.save(str(path)+'/hinfo', hinfo)

In [ ]:
def p(x):
    return 1/(1.*x)
def collapse(gv, NC): 
    na = np.shape(gv)[2]//NC
    Nu = np.shape(gv)[0]
    cgv = np.zeros((Nu,Nu,na), dtype='complex128')
    bb=0
    for ii in range(na):
        aa = bb
        bb = aa+NC
        cgv[:,:,ii] = np.mean(gv[:,:,aa:bb], axis=-1)  
    return cgv

In [ ]:
FWHM, dUg, f, nua = np.load(str(path)+'hinfo.npy', allow_pickle=True)
# U1, U2, NBin = 5, 200, 15
print(FWHM, dUg, f, nua)

# Binning in planes (Annular rings on planes)

In [ ]:
du_ring=(Nmax-1)/(1.*Nr) 
uLow=np.array([i*du_ring+1 for i in range(Nr)]  )
uHigh=np.array([(i+1)*du_ring+1 for i in range(Nr)])
uavg=(2/3)*(uHigh**3-uLow**3)/(uHigh**2-uLow**2)

In [ ]:
uavg

# Data

# Sky data (not visibility). Use this or GV data. Dont use both

In [ ]:
### sky signal ####
# loc='/home/sukhdeep/Documents/Research/Bispectrum/Visibility_Bispectrum/Visibility_Correlation/asif_da/1_ngrf/output_data/2D_ngrf_field_R1'

# NN = 1024  # Example value, replace with the actual NN value
# num_elements = NN * (NN // 2 + 1) * 2  # Two doubles (real and imaginary) per element
# data = np.fromfile(loc, dtype=np.float64, count=num_elements)
# # reshaped_data = data.reshape(NN, (NN // 2 + 1), 2)
# imag_part = data[1::2]
# real_part = data[::2]
# real_part=real_part.reshape(NN, (NN // 2 + 1))
# imag_part=imag_part.reshape(NN, (NN // 2 + 1))

# # real_part = reshaped_data[:, :, 0]
# # imag_part = reshaped_data[:, :, 1]
# complex_array = real_part + 1j * imag_part
# full_complex_array = np.zeros((NN, NN), dtype=np.complex128)
# full_complex_array[:, :(NN // 2 + 1)] = complex_array

# for i in range(NN):
#     for j in range(1, NN // 2):
#         full_complex_array[-i, -j] = np.conj(full_complex_array[i, j])


# This is just for future ref how FFT behaves

In [ ]:
### following is same as complex_array but more intiutive #########

# rep = np.zeros((NN, NN//2+1))
# imp = np.zeros((NN, NN//2+1))
# for i in range(NN):
#     for j in range( NN // 2+1):
#         index = i*(NN//2+1)+j
#         rep[i][j]= data[::2][index]
#         imp[i][j]= data[1::2][index]
# comp_arr = rep + 1j * imp

###### following all are same #########
# np.std(np.fft.ifft2(full_complex_array)/np.fft.irfft2(complex_array))
# np.std(np.fft.irfft2(comp_arr)/np.fft.irfft2(complex_array))


### just for checking symmteries hermitiaan #########
# xx = np.random.rand(6, 6 )
# xxu=np.fft.fft2(xx)
# i=2
# j=4
# xxu[-i, -j],np.conj(xxu[i,j])


####### following symmteries are hermittin################
# full_complex_array[0, 0] = np.real(full_complex_array[0, 0])  # Zero frequency
# full_complex_array[NN // 2, 0] = np.real(full_complex_array[NN // 2, 0])  # Nyquist frequency in the first row
# full_complex_array[0, NN // 2] = np.real(full_complex_array[0, NN // 2])  # Nyquist frequency in the first column
# full_complex_array[NN // 2, NN // 2] = np.real(full_complex_array[NN // 2, NN // 2])  # Nyquist frequency in the center

# for i in range(1, NN // 2):
#     full_complex_array[-i, 0] = np.conj(full_complex_array[i, 0])
#     full_complex_array[0, -i] = np.conj(full_complex_array[0, i])
# for i in range(1, NN):
#     for j in range(NN // 2 + 1, NN):
#         full_complex_array[-i, -j] = np.conj(full_complex_array[i, j])

# GV data not sky signal. Use this for 3-visi-corr

In [ ]:
##### GV data #############
filename=str(path)+'GV_data_1'
headersize = (nc+3)*np.float64().itemsize
aa = np.fromfile(filename, dtype='float64', count=headersize, offset=0)      
FWHM, dUg, f, nua = aa[0], aa[1], aa[2], aa[3:nc+3] # header
hinfo = (FWHM, dUg, f, nua)
aa = np.fromfile(filename, dtype='complex128', count=-1, offset=headersize)
Nu = int(np.sqrt(len(aa)/(3*nc)))
NN = Nu*Nu*nc
GR = collapse(np.reshape(aa[:NN], (Nu, Nu, nc))[xn:Nu-xn, xn:Nu-xn, :], NC)
GL = collapse(np.reshape(aa[NN:2*NN], (Nu, Nu, nc))[xn:Nu-xn, xn:Nu-xn, :], NC)
AA = collapse(np.reshape(aa[2*NN:3*NN], (Nu, Nu, nc))[xn:Nu-xn, xn:Nu-xn, :], NC)

In [ ]:
np.shape(GR)

In [ ]:
NN

In [ ]:
plt.imshow(np.real(GL[:,:,0]),extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# plt.xlim(-100,100)
# plt.ylim(-100,100)
plt.colorbar()

In [ ]:
# field=np.zeros((Nu,Nu))
# N1=Nu
# N2=Nu
# for i in range(N1):
#     for j in range(N2):
#         ic = i+int(N2/2);
#         jc = j+int(N2/2);
#         iagv=(ic-N1) if (ic>=N1) else ic ;
#         index = int(iagv)*N1+jc
# #         if jgv>992:
# #         print(int(index))
#         field[i][j] = np.real(aa[int(index)])

In [ ]:
# plt.imshow(field,extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# # plt.xlim(-500,-400)
# # plt.ylim(-500,-400)
# plt.colorbar()

# Calculating grid points in different rings and storing its IFFT in array

In [ ]:
# ## either use this or shift the origin before fft ###########
# Nu=500
# N1 = Nu
# N2 = Nu
# points_in_ring=np.zeros((Nr,N1,N2),np.clongdouble)
# for ring in range(Nr):
#     for i in range(N1):
#         for j in range(int(N2)):
#             ia=i-N1 if i>N1/2 else i
#             ja=j-N2 if j>N2/2 else j
#             kk=np.sqrt(ia*ia+ja*ja)
#             if (kk>=uLow[ring] and kk<uHigh[ring] and kk>0):
#                 points_in_ring[ring][i][j]=1.0+0j

# II=np.array([N1*N2*np.fft.ifft2(points_in_ring[i]) for i in range(Nr)])

In [ ]:
# Nu = 92 # use if using sky signal
N1 = Nu
N2 = Nu
Nm = (Nu)//2# (Nu-1)//2
tpibyL=2.*np.pi/LL; 
Cx=2.*np.pi/(N1*LL);  Cy=2.*np.pi/(N2*LL);
Area=(N1*LL)*(N2*LL);

points_in_ring=np.zeros((Nr,N1,N2),np.clongdouble)
for ring in range(Nr):
    for i in range(N1):
        for j in range(int(N2)):
            kk=np.sqrt((i-Nm)*(i-Nm)+(j-Nm)*(j-Nm))#*dUg
            if (kk>=uLow[ring] and kk<uHigh[ring] and kk>0):
                points_in_ring[ring][i][j]=1.0+0j
                
shifted_origin = np.array([np.fft.ifftshift(points_in_ring[i]) for i in range(Nr)])

# II=np.array([N1*N2*np.fft.ifft2(points_in_ring[i]) for i in range(Nr)]) 
II=np.array([N1*N2*np.fft.ifft2(shifted_origin[i]) for i in range(Nr)])

In [ ]:
924//2

In [ ]:
xx=np.real(points_in_ring[7])*np.real(GL[:,:,0])

In [ ]:
# plt.imshow(np.real(points_in_ring[7]),extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
plt.imshow(np.real(II[1]),extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# plt.imshow(xx,extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# plt.xlim(-100,100)
# plt.ylim(-100,100)
plt.colorbar()

# Calculating and storing number of triangles for each possible combination of 3 rings

In [ ]:
###### Calculating possible combinations of closed triangles
rng=[]
q11=0

m1_arr=[]
m2_arr=[]
m3_arr=[]
for m1 in range(Nr):
    for m2 in range(m1+1):
        tt=uavg[m2]/uavg[m1]
        if (tt>=0.5 and tt<=1.):
            for m3 in range(m2+1):
                mu=0.5*(tt+1./tt-(uavg[m3]/uavg[m1])*(uavg[m3]/uavg[m2]))
                if (mu>=0.5/tt and mu<=1.):
                    m1_arr.append(m1)
                    m2_arr.append(m2)
                    m3_arr.append(m3)  

# Triangles

In [ ]:
##### Calculating number of closed triangles corresponding to each combination calculated above
triangle=[]
for rl in range(len(m1_arr)):
    m1=m1_arr[rl]
    m2=m2_arr[rl]
    m3=m3_arr[rl]
                    
    I1=II[m1] 
    I2=II[m2]
    I3=II[m3]
    Num=np.real(np.sum(I1*I2*I3))
    triangle.append(Num)

# Bispectrum

In [ ]:
##### sky signal bispectrum. Use this snippet or use the below snippet. Dont use both. ##########

# shifted_sky_data = np.fft.fftshift(full_complex_array)

# sky_data = np.array([np.fft.ifftshift(points_in_ring[i]*shifted_sky_data) for i in range(Nr)])
 
# DD=np.array([N1*N2*np.fft.ifft2(sky_data[i]) for i in range(Nr)])

# visibility bispectrum

In [ ]:
vis_data = np.array([np.fft.ifftshift(points_in_ring[i]*GL[:,:,0]) for i in range(Nr)])
 
DD=np.array([N1*N2*np.fft.ifft2(vis_data[i]) for i in range(Nr)])

In [ ]:
plt.imshow(np.real(vis_data[5]),extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# plt.imshow(np.real(II[1]),extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# plt.imshow(xx,extent=[-500,500,-500,500],origin='lower',cmap='RdBu')
# plt.xlim(-100,100)
# plt.ylim(-100,100)
plt.colorbar()

In [ ]:
bispectrum = []
for rl in range(len(m1_arr)):
    m1=m1_arr[rl]
    m2=m2_arr[rl]
    m3=m3_arr[rl]
                    
    D1=DD[m1] 
    D2=DD[m2]
    D3=DD[m3]
    bs=np.real(np.sum(D1*D2*D3))
    bispectrum.append(bs)

In [ ]:
# bs_res = np.array(bispectrum)/triangle
ntri   = np.array(triangle)/N1/N2

In [ ]:
ntri[0]/6.079387e+06

In [ ]:
ntri

In [ ]:
ntri

In [ ]:
Nu